This notebook demonstrates how to use the BiomechZoo library to analyze movement data collected from IMU sensors.
The analysis includes data preprocessing, event detection, gait cycle partitioning, normalization, and visualization of the results.

Data source: The data used in this notebook is from the "Losing" dataset, which contains IMU data collected from participants performing
various walking tasks. The data were modified to simplify the analysis.

The dataset is available at: https://doi.org/10.6084/m9.figshare.c.5758997

The dataset is described in the following paper: Losing, V., Hasenjäger, M. A Multi-Modal Gait Database of Natural Everyday-Walk in an Urban Environment. Sci Data 9, 473 (2022). https://doi.org/10.1038/s41597-022-01580-3

In [65]:
# import libraries
import os
from biomechzoo.biomechzoo import BiomechZoo
from biomechzoo.utils.find_repo_root import find_repo_root

In [72]:
# Step 0: get raw data folder and initialize biomechzoo object
project_root = find_repo_root(os.getcwd())
fld_raw_data = os.path.join(project_root, 'data', 'losing', 'raw')
inplace = False  # tells biomechzoo to create new folders instead of writing to the original folder
verbose = 1 # minimal output to the user
bmech = BiomechZoo(fld_raw_data, verbose=verbose, inplace=inplace)

BiomechZoo initialized
verbosity set to: 1
root processing folder set to: /Users/phild/Code/biomechzoo/data/losing/raw
Processing mode: backup (inplace=False)(each step will be applied to a new folder)


In [73]:
# step 1: convert to zoo
# frequency is 60 Hz as per the database documentation
bmech.table2zoo(extension='csv', freq=60, out_folder='1-csv2zoo')

table2zoo process complete for 1 file(s) in 3.01 secs
all files saved to: /Users/phild/Code/biomechzoo/data/losing/1-csv2zoo


In [74]:
# step 2 clean
ch_remove = ['angularVelocity_Pelvis_x', 'angularVelocity_Pelvis_y','angularVelocity_Pelvis_z', 'participant_id',
             'task', 'walk_mode', 'section']
ch_old = ['jointAngle_jRightKnee_x', 'jointAngle_jRightKnee_y', 'jointAngle_jRightKnee_z']
ch_new = ['RightKneeAngle_x', 'RightKneeAngle_y', 'RightKneeAngle_z']
bmech.removechannel(ch=ch_remove, out_folder='2-clean')
bmech.renamechannel(ch_current=ch_old, ch_new=ch_new, out_folder='2-clean', inplace=True)

Removing channels: ['participant_id', 'task', 'angularVelocity_Pelvis_x', 'angularVelocity_Pelvis_y', 'angularVelocity_Pelvis_z', 'walk_mode', 'section']
removechannel process complete for 1 file(s) in 0.24 secs
all files saved to: /Users/phild/Code/biomechzoo/data/losing/2-clean
renamechannel process complete for 1 file(s) in 0.02 secs
all files saved to: /Users/phild/Code/biomechzoo/data/losing/2-clean


In [75]:
# inspect the file
bmech.file_info()

File: /Users/phild/Code/biomechzoo/data/losing/2-clean/honda_walk_subset_p1.zoo
Size: 5846080 bytes
Video frequency: 60 Hz
Analog frequency: {} Hz
Channels (13): time, acceleration_Pelvis_x, acceleration_Pelvis_y, acceleration_Pelvis_z, acceleration_RightLowerLeg_x, acceleration_RightLowerLeg_y, acceleration_RightLowerLeg_z, angularVelocity_RightLowerLeg_x, angularVelocity_RightLowerLeg_y, angularVelocity_RightLowerLeg_z, RightKneeAngle_x, RightKneeAngle_y, RightKneeAngle_z
Samples: 56151
First 5 sample(s):
sample	time	acceleration_Pelvis_x	acceleration_Pelvis_y	acceleration_Pelvis_z	acceleration_RightLowerLeg_x	acceleration_RightLowerLeg_y	acceleration_RightLowerLeg_z	angularVelocity_RightLowerLeg_x	angularVelocity_RightLowerLeg_y	angularVelocity_RightLowerLeg_z	RightKneeAngle_x	RightKneeAngle_y	RightKneeAngle_z
0	0	-3.558701	-1.303026	-0.356919	15.636838	-2.487923	-4.821456	0.501537	2.840339	-0.429778	1.090504	-0.525838	16.094898
1	16	-3.47593	1.365	-0.573515	14.044996	-3.616945	-0.1

In [79]:
# step 3: get foot strikes from Kielmat algorithm
bmech.compute_magnitude(ch_1='acceleration_RightLowerLeg_x', ch_2='acceleration_RightLowerLeg_y', ch_3='acceleration_RightLowerLeg_z', ch_new='acceleration_RightLowerLeg_magnitude', out_folder='3-processing', inplace=True)
bmech.addevent(event_type='mcgrath_fs', event_name='foot_strike', ch='acceleration_RightLowerLeg_magnitude', out_folder='3-processing', inplace=True)

compute_magnitude process complete for 1 file(s) in 0.05 secs
all files saved to: /Users/phild/Code/biomechzoo/data/losing/2-clean
addevent process complete for 1 file(s) in 0.11 secs
all files saved to: /Users/phild/Code/biomechzoo/data/losing/2-clean


In [80]:
# step 4: partition by gait cycle and normalize
bmech.split_trial_by_gait_cycle(first_event_name='foot_strike_1', out_folder='4-split', inplace=False)
bmech.normalize()

split_trial_by_gait_cycle process complete for 1 file(s) in 52.65 secs
all files saved to: /Users/phild/Code/biomechzoo/data/losing/4-split
normalize process complete for 957 file(s) in 56.95 secs
all files saved to: /Users/phild/Code/biomechzoo/data/losing/processed


In [ ]:
# inspect one file
bmech.plot(ch='RightKneeAngle_z', file_path='honda_walk_subset_p1_6.zoo')


In [ ]:
# step 7 plot all files
import os
# import statements
from biomechzoo.ensembler.ensembler import Ensembler
from biomechzoo.ensembler.plot_spec import PlotSpec
from biomechzoo.ensembler.renderers import IndividualLinesRenderer, MeanSDRenderer, CompositeRenderer
from biomechzoo.ensembler.helpers import ConditionSpec, ConditionSource

spec = ConditionSpec(source = ConditionSource.BETWEEN,
                     conditions = ['running'])
channels = ['acc_mag']
rows = 1
cols = 2

lines_and_events = CompositeRenderer(IndividualLinesRenderer())     # within stuff

fig = (
    Ensembler(in_folder=bmech.in_folder,  channels=channels, n_rows=rows,  n_cols=cols, condition_spec=spec)
    .add_subplot(PlotSpec(channel='acc_mag', condition = 'running', companions = [], row=1, col=1, renderer=lines_and_events))
    .add_subplot(PlotSpec(channel='acc_mag', condition = 'running', companions = [], row=1, col=2, renderer=MeanSDRenderer()))
    .build(title=' running')
)
fig.show()


